In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mxlpy import (
    OdeModelBuilder,
    Simulator,
    cartesian_product,
    fit,
    make_protocol,
    plot,
    scan,
)
from mxlpy.meta import generate_model_code_mxlweb

# ODE models

*mxlpy* has three model builders, which differ in how the right hand side of the system is written down:

| Builder | Right hand side | Typical use |
|:--|:--|:--|
| `KineticModelBuilder` (`Model`) | $N \cdot v(x)$: reactions with rate laws and a stoichiometric matrix | metabolic & signalling networks |
| `OdeModelBuilder` | $\frac{dx}{dt} = f(x)$: one differential equation per variable | phenomenological models, equations taken from a paper |
| `SteadyStateModelBuilder` | no variables, only explicit functions of parameters | algebraic / steady-state models, see [Steady-state models](steady-state-model-builder.ipynb) |

This notebook covers the `OdeModelBuilder`.
Use it when your model is naturally written as differential equations rather than as a reaction network.
Since there are no reactions, an `OdeModelBuilder` has no fluxes and no stoichiometries.

As an example we use logistic growth with a constant harvesting rate $h$

$$\frac{dx}{dt} = r x \left(1 - \frac{x}{K}\right) - h x$$

which has the non-trivial steady state $x^* = K \left(1 - \frac{h}{r}\right)$.

## Building the model

Instead of `add_reaction`, every variable gets exactly one differential equation via `add_diff_eq`, which also sets the variable's initial value.
Parameters and derived quantities work just like in the `KineticModelBuilder`.

As with all builders, use named functions instead of lambdas (they need to be inspectable and pickle-able).

In [ ]:
def logistic_harvest(x: float, r: float, capacity: float, h: float) -> float:
    return r * x * (1 - x / capacity) - h * x


def fraction(x: float, capacity: float) -> float:
    return x / capacity


def get_model() -> OdeModelBuilder:
    return (
        OdeModelBuilder()
        .add_parameters({"r": 1.0, "capacity": 10.0, "h": 0.2})
        .add_diff_eq(
            "x",
            fn=logistic_harvest,
            args=["x", "r", "capacity", "h"],
            initial_value=0.5,
        )
        .add_derived("saturation", fn=fraction, args=["x", "capacity"])
    )


model = get_model()
model.diff_eqs

In [ ]:
model.derived

The right hand side can be evaluated directly, either at the initial conditions or at custom values

In [ ]:
model.get_right_hand_side({"x": 5.0})

Initial values can be changed with `update_variable` / `update_variables`

In [ ]:
get_model().update_variable("x", 2.0).get_initial_conditions()

## Simulation

An `OdeModelBuilder` is simulated with the same `Simulator` as a `KineticModelBuilder`.
The result is an `OdeSimulation`, which offers the same access patterns as a `Simulation`, minus everything flux related.

In [ ]:
res = Simulator(get_model()).simulate(t_end=15).get_result().unwrap_or_err()

fig, (ax1, ax2) = plot.two_axes(figsize=(7, 3))
plot.lines(res.variables, ax=ax1)
plot.lines(res.get_right_hand_side(), ax=ax2)
ax1.set(xlabel="time / a.u.", ylabel="value / a.u.")
ax2.set(xlabel="time / a.u.", ylabel="dx/dt / a.u.")
plt.show()

`get_combined` returns all variables, derived variables and readouts as a single `pandas.DataFrame`.
`get_args` additionally allows including parameters and derived parameters.

In [ ]:
res.get_args(include_parameters=True).tail()

Steady states and protocols work as usual.
For the default parameters the steady state is at $x^* = 10 \cdot (1 - 0.2) = 8$

In [ ]:
Simulator(get_model()).simulate_to_steady_state().get_result().unwrap_or_err().get_new_y0()

In [ ]:
protocol = make_protocol(
    [
        (10, {"h": 0.2}),
        (10, {"h": 0.6}),  # overharvesting
        (10, {"h": 0.2}),
    ]
)
res = (
    Simulator(get_model())
    .simulate_protocol(protocol, time_points_per_step=20)
    .get_result()
    .unwrap_or_err()
)

fig, ax = plot.lines(res.get_variables(include_derived_variables=False))
plot.shade_protocol(protocol["h"], ax=ax, alpha=0.1)
ax.set(xlabel="time / a.u.", ylabel="x / a.u.")
plt.show()

## Parameter scans

All functions of the `scan` module accept an `OdeModelBuilder`.
Their results have the same shape as for a `KineticModelBuilder`, just without fluxes.

In [ ]:
res = scan.steady_state(
    get_model(),
    to_scan=pd.DataFrame({"h": np.linspace(0, 0.9, 10)}),
)

fig, ax = plot.lines(res.variables)
ax.set(xlabel="harvesting rate h / a.u.", ylabel="steady state / a.u.")
plt.show()

Scanning multiple parameters works via `cartesian_product`

In [ ]:
res = scan.steady_state(
    get_model(),
    to_scan=cartesian_product(
        {
            "h": np.linspace(0, 0.9, 10),
            "capacity": np.linspace(5, 15, 5),
        }
    ),
)

fig, ax, _ = plot.heatmap_from_2d_idx(res.variables, "x")
ax.set(xlabel="h", ylabel="capacity", title="steady state of x")
plt.show()

The `to_scan` DataFrame can also contain initial conditions

In [ ]:
res = scan.time_course(
    get_model(),
    to_scan=pd.DataFrame({"x": [0.1, 1.0, 5.0, 15.0]}),
    time_points=np.linspace(0, 10, 101),
)

fig, ax = plot.lines(res.get_by_name("x"))
ax.set(xlabel="time / a.u.", ylabel="x / a.u.")
plt.show()

## Fitting

The single-model routines `fit.steady_state`, `fit.time_course` and `fit.protocol_time_course` accept an `OdeModelBuilder`.
The returned `Fit` contains a copy of the `OdeModelBuilder`.

Here we create some synthetic data with known parameters and fit the model to it.

In [ ]:
data = (
    Simulator(get_model().update_parameters({"r": 0.7, "h": 0.35}))
    .simulate_time_course(np.linspace(0, 15, 31))
    .get_result()
    .unwrap_or_err()
    .get_variables(
        include_derived_variables=False,
        include_readouts=False,
        include_surrogate_variables=False,
    )
)

fitted = fit.time_course(
    get_model(),
    p0={"r": 1.0, "h": 0.2},
    data=data,
    minimizer=fit.LocalScipyMinimizer(),
).unwrap_or_err()
fitted.best_pars

In [ ]:
res = (
    Simulator(fitted.model.update_parameters(fitted.best_pars))
    .simulate_time_course(data.index)
    .get_result()
    .unwrap_or_err()
)

fig, ax = plot.lines(res.get_variables(include_derived_variables=False))
ax.scatter(data.index, data["x"], color="black", label="data")
ax.set(xlabel="time / a.u.", ylabel="x / a.u.")
ax.legend()
plt.show()

## Export

`OdeModelBuilder` models can be saved to the native `.mxl.json` format with `mxlpy.save` / `mxlpy.load`, and exported to [MxlWeb](https://github.com/Computational-Biology-Aachen/mxl-web), which natively supports differential equation models.

In [ ]:
print(generate_model_code_mxlweb(get_model()))

## Current limitations

The `OdeModelBuilder` doesn't yet support everything the `KineticModelBuilder` does.
Not supported are

- everything flux or stoichiometry related, e.g. metabolic control analysis (`mca`) and label models
- Monte Carlo methods (`mc`)
- group, ensemble, joint and carousel fits
- Jacobians (`Simulator(..., use_jacobian=True)` falls back to finite differences with a warning) and events